# Soluciones — Clase 17: Estadística inferencial — pruebas de hipótesis

Este notebook contiene soluciones de referencia para los ejercicios de la clase.
Intenta resolverlos por tu cuenta antes de consultar estas soluciones.

> El objetivo no es memorizar el código, sino aprender a formular
> hipótesis y leer los resultados con criterio.

In [ ]:
# Configuración inicial
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted")

transporte = pd.read_csv("../../datasets/transporte.csv")
estudiantes = pd.read_csv("../../datasets/estudiantes.csv")

print("Datos cargados.")
print("Columnas transporte:", transporte.columns.tolist())
print("Columnas estudiantes:", estudiantes.columns.tolist())

In [ ]:
# SOLUCIÓN — Ejercicios 1 y 2: Descriptiva + visualización previa

# Separar grupos
con_lluvia = transporte[transporte["lluvia"] == True]["retraso_min"].dropna()
sin_lluvia = transporte[transporte["lluvia"] == False]["retraso_min"].dropna()

# Estadística descriptiva
print("=== ESTADÍSTICA DESCRIPTIVA ===")
resumen = pd.DataFrame({
    "Con lluvia": con_lluvia.describe(),
    "Sin lluvia": sin_lluvia.describe()
}).round(2)
print(resumen)
print(f"\nDiferencia de medias: {con_lluvia.mean() - sin_lluvia.mean():.2f} min")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=transporte, x="lluvia", y="retraso_min",
            palette=["lightblue", "steelblue"], ax=axes[0])
axes[0].set_title("Retraso por condición de lluvia")
axes[0].set_xticklabels(["Sin lluvia", "Con lluvia"])
axes[0].set_ylabel("Retraso (min)")

# Histogramas superpuestos para comparar distribuciones
sns.histplot(con_lluvia, color="steelblue", alpha=0.6, label="Con lluvia", kde=True, ax=axes[1])
sns.histplot(sin_lluvia, color="lightcoral", alpha=0.6, label="Sin lluvia", kde=True, ax=axes[1])
axes[1].set_title("Distribuciones superpuestas")
axes[1].set_xlabel("Retraso (min)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# SOLUCIÓN — Ejercicio 3 y 4: Prueba t con alfa = 0.05 y alfa = 0.01

# H0: media con lluvia == media sin lluvia
# H1: media con lluvia != media sin lluvia

t_stat, p_value = stats.ttest_ind(con_lluvia, sin_lluvia)

print("=== PRUEBA T DE STUDENT ===")
print(f"Estadístico t:  {t_stat:.4f}")
print(f"p-value:        {p_value:.6f}")
print()

for alfa in [0.05, 0.01]:
    if p_value < alfa:
        print(f"Con alfa = {alfa}: RECHAZAMOS H0. Diferencia estadísticamente significativa.")
    else:
        print(f"Con alfa = {alfa}: NO rechazamos H0. No hay evidencia suficiente.")

print()
print("Conclusión en lenguaje simple:")
if p_value < 0.05:
    print(f"Los datos muestran que los retrasos SON más largos cuando llueve ({con_lluvia.mean():.1f} min")
    print(f"vs {sin_lluvia.mean():.1f} min sin lluvia). Esta diferencia es muy poco probable")
    print("que sea producto del azar — hay evidencia real de que la lluvia afecta los retrasos.")
else:
    print("No hay suficiente evidencia para afirmar que la lluvia causa más retrasos.")

In [ ]:
# SOLUCIÓN — Ejercicio 5: Intervalos de confianza

def ic_95(datos):
    n = len(datos)
    media = datos.mean()
    se = stats.sem(datos)
    inf, sup = stats.t.interval(0.95, df=n-1, loc=media, scale=se)
    return media, inf, sup

m_ll, inf_ll, sup_ll = ic_95(con_lluvia)
m_sl, inf_sl, sup_sl = ic_95(sin_lluvia)

print("=== INTERVALOS DE CONFIANZA DEL 95% ===")
print(f"Con lluvia:  {m_ll:.2f} min  |  IC = [{inf_ll:.2f}, {sup_ll:.2f}]")
print(f"Sin lluvia:  {m_sl:.2f} min  |  IC = [{inf_sl:.2f}, {sup_sl:.2f}]")
print()

# Verificar superposición
if sup_sl < inf_ll or sup_ll < inf_sl:
    print("Los intervalos NO se superponen => diferencia significativa.")
else:
    print("Los intervalos SE superponen => la diferencia no es clara.")

# Visualización de intervalos
plt.figure(figsize=(7, 4))
grupos = ["Con lluvia", "Sin lluvia"]
medias = [m_ll, m_sl]
errores = [[m_ll - inf_ll, m_sl - inf_sl], [sup_ll - m_ll, sup_sl - m_sl]]

plt.bar(grupos, medias, color=["steelblue", "lightcoral"], alpha=0.8, width=0.4)
plt.errorbar(grupos, medias, yerr=[[m_ll - inf_ll, m_sl - inf_sl], [sup_ll - m_ll, sup_sl - m_sl]],
             fmt="none", color="black", capsize=6, linewidth=2)
plt.title("Retraso promedio con IC del 95%")
plt.ylabel("Retraso (minutos)")
plt.tight_layout()
plt.show()

In [ ]:
# SOLUCIÓN — Ejercicios 6 y 7: Prueba t entre cursos + Chi-cuadrado

cursos = estudiantes["curso"].unique()
curso_a, curso_b = cursos[0], cursos[1]

notas_a = estudiantes[estudiantes["curso"] == curso_a]["nota_final"].dropna()
notas_b = estudiantes[estudiantes["curso"] == curso_b]["nota_final"].dropna()

t2, p2 = stats.ttest_ind(notas_a, notas_b)
print(f"Prueba t entre {curso_a} y {curso_b}:")
print(f"  t = {t2:.4f}, p = {p2:.4f}")
print(f"  => {'Diferencia significativa' if p2 < 0.05 else 'Sin diferencia significativa'}")
print()

# Chi-cuadrado: ¿el curso afecta la probabilidad de aprobar?
estudiantes["aprobado"] = (estudiantes["nota_final"] >= 60).map({True: "Aprobado", False: "Reprobado"})
tabla = pd.crosstab(estudiantes["curso"], estudiantes["aprobado"])
chi2, p_chi, gl, _ = stats.chi2_contingency(tabla)

print("Prueba chi-cuadrado (curso vs aprobado):")
print(f"  chi2 = {chi2:.4f}, p = {p_chi:.4f}, gl = {gl}")
print(f"  => {'Relación significativa entre curso y aprobación' if p_chi < 0.05 else 'No hay relación significativa'}")
print()
print("Tabla de porcentajes de aprobación por curso:")
print(tabla.div(tabla.sum(axis=1), axis=0).mul(100).round(1))

In [ ]:
# SOLUCIÓN — Ejercicio 8: Reflexión sobre errores

print("=== REFLEXIÓN SOBRE ERRORES TIPO I Y TIPO II ===")
print()
print("1. Si p = 0.04 y concluimos que hay diferencia real, podríamos estar cometiendo")
print("   un ERROR TIPO I (falsa alarma). Hay un 4% de probabilidad de que el resultado")
print("   sea producto del azar aunque no haya diferencia real.")
print()
print("2. En medicina, es más grave no detectar una enfermedad (error tipo II) que")
print("   dar una falsa alarma (error tipo I). Se minimiza el error tipo II.")
print("   Esto implica usar un alfa más alto (0.10 o más) para ser más sensible.")
print()
print("3. Con 20 pruebas y alfa = 0.05:")
positivos_por_azar = 20 * 0.05
print(f"   Se esperan ~{positivos_por_azar:.0f} resultado(s) positivo(s) solo por azar.")
print("   Por eso en investigación se aplica la corrección de Bonferroni u otras")
print("   cuando se hacen muchas pruebas sobre el mismo conjunto de datos.")

## Reflexión final

La estadística inferencial no da certezas absolutas: da probabilidades.
Cuando reportamos p < 0.05, estamos diciendo que si repitiéramos el experimento
muchas veces con datos diferentes, menos del 5% de las veces obtendríamos
este resultado por azar.

Eso es poderoso — pero también significa que debemos ser honestos:
siempre existe la posibilidad de equivocarse. La ciencia avanza precisamente
porque esa posibilidad se puede medir.

En la próxima clase aprenderemos a preparar las variables para los modelos
de machine learning — el paso natural después de entender los datos estadísticamente.